# Data Loading

## Loading NVD Data into Neo4j

We first implement the necessary code to load NVD CVE data into the Neo4j graph database. This includes parsing the JSON files, creating nodes for CVEs, and establishing relationships with other entities in the graph.

In this implementation, we assume you have cloned the NVD Git repository locally with:

```bash
git clone https://github.com/CVEProject/cvelistV5.git
```

You could also change this implementation to pull data directly from the NVD API if preferred - we picked the local file approach for simplicity and speed.

In [ ]:
import json
import os
from pathlib import Path
from neo4j import GraphDatabase
from dotenv import load_dotenv
from tqdm.auto import tqdm

load_dotenv()

# Connection details
URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
AUTH = (os.getenv("NEO4J_USER", "neo4j"), os.getenv("NEO4J_PASSWORD", "password"))
DB = os.getenv("NEO4J_DB", "nvd")
NVD_REPO_PATH = os.getenv("NVD_REPO_PATH", "./cvelistV5/cves/")
KEV_URL = os.getenv("KEV_URL", "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json")

def create_constraints():
    queries = [
        "CREATE CONSTRAINT cve_id_unique IF NOT EXISTS FOR (c:CVE) REQUIRE c.id IS UNIQUE",
        "CREATE CONSTRAINT cwe_id_unique IF NOT EXISTS FOR (w:CWE) REQUIRE w.id IS UNIQUE",
        "CREATE CONSTRAINT vendor_name_unique IF NOT EXISTS FOR (v:Vendor) REQUIRE v.name IS UNIQUE",
        "CREATE CONSTRAINT product_name_unique IF NOT EXISTS FOR (p:Product) REQUIRE p.name IS UNIQUE"
    ]
    
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        with driver.session(database=DB) as session:
            for query in queries:
                try:
                    session.run(query)
                except Exception as e:
                    print(f"Failed to create constraint: {e}")

def load_cve_batch(tx, batch):
    query = """
    UNWIND $batch AS data
    WITH data, data.containers.cna AS cna, data.containers.adp AS adp_list, data.cveMetadata AS cveMetadata
    
    // Create/Update CVE Node
    MERGE (v:CVE {id: data.cveMetadata.cveId})
    SET v.state = data.cveMetadata.state,
        v.assigner = data.cveMetadata.assignerShortName,
        v.publishedDate = datetime(data.cveMetadata.datePublished),
        v.lastModifiedDate = datetime(data.cveMetadata.dateUpdated),
        v.source = data.cveMetadata.source,
        v.description = [d IN cna.descriptions WHERE d.lang = 'en'][0].value
    
    // Metrics Logic (Consolidated CNA and ADP)
    // We keep 'cna' and 'adp_list' in scope for subsequent sections
    WITH v, cna, adp_list,
         coalesce(cna.metrics, []) + 
         reduce(acc = [], x IN coalesce(adp_list, []) | acc + coalesce(x.metrics, [])) AS all_metrics
    
    WITH v, cna, adp_list, [m IN all_metrics WHERE m.cvssV3_1 IS NOT NULL OR m.cvssV3_0 IS NOT NULL][0] AS metric
    WITH v, cna, adp_list, coalesce(metric.cvssV3_1, metric.cvssV3_0) AS cvss
    
    FOREACH (_ IN CASE WHEN cvss IS NOT NULL THEN [1] ELSE [] END |
        SET v.baseScore = cvss.baseScore,
            v.baseSeverity = cvss.baseSeverity,
            v.attackComplexity = cvss.attackComplexity,
            v.attackVector = cvss.attackVector,
            v.availabilityImpact = cvss.availabilityImpact,
            v.confidentialityImpact = cvss.confidentialityImpact,
            v.integrityImpact = cvss.integrityImpact,
            v.privilegesRequired = cvss.privilegesRequired,
            v.userInteraction = cvss.userInteraction
    )

    // Vendors and Products (Consolidated CNA and ADP)
    // Combine 'affected' lists from CNA and all ADP entries
    WITH v, cna,
         coalesce(cna.affected, []) + 
         reduce(acc = [], x IN coalesce(adp_list, []) | acc + coalesce(x.affected, [])) AS all_affected
    
    UNWIND all_affected AS affected
    WITH v, cna, affected 
    WHERE affected.product IS NOT NULL AND affected.product <> 'n/a'
      AND affected.vendor IS NOT NULL AND affected.vendor <> 'n/a'
    
    MERGE (vend:Vendor {name: affected.vendor})
    MERGE (p:Product {name: affected.product})
    MERGE (vend)-[:PROVIDES]->(p)
    MERGE (v)-[:AFFECTS]->(p)

    // Problem Types (CWEs)
    WITH v, cna
    UNWIND coalesce(cna.problemTypes, []) AS pt
    UNWIND coalesce(pt.descriptions, []) AS desc
    WITH v, desc WHERE (desc.cweId IS NOT NULL AND desc.cweId =~ 'CWE-\\\\d+') 
                 OR (desc.description IS NOT NULL AND desc.description =~ 'CWE-\\\\d+')
    MERGE (w:CWE {id: coalesce(desc.cweId, desc.description)})
    MERGE (v)-[:HAS_PROBLEM_TYPE]->(w)
    """
    tx.run(query, batch=batch)

def process_local_repo(years: list = None):
    base_path = Path(NVD_REPO_PATH)
    batch_size = 1000
    current_batch = []
    total_processed = 0

    # Pre-collect filtered files to get a total count
    print("Scanning directory for files...")
    all_files = list(base_path.rglob("CVE-*.json"))
    if years:
        all_files = [f for f in all_files if any(str(y) in f.parts for y in years)]
    
    total_files = len(all_files)

    # Process with a full progress bar
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        with driver.session(database=DB) as session:
            # Wrap the list in tqdm for a % completion bar
            for json_file in tqdm(all_files, desc="Ingesting CVEs", unit="cve"):
                try:
                    with open(json_file, 'r', encoding='utf-8') as f:
                        data = json.load(f)
                        if data.get("cveMetadata", {}).get("state") == "PUBLISHED":
                            current_batch.append(data)
                except Exception as e:
                    # Use tqdm.write instead of print to avoid breaking the bar
                    tqdm.write(f"Error in {json_file.name}: {e}")

                if len(current_batch) >= batch_size:
                    session.execute_write(load_cve_batch, current_batch)
                    total_processed += len(current_batch)
                    current_batch = []

            if current_batch:
                session.execute_write(load_cve_batch, current_batch)
                total_processed += len(current_batch)

We also implement a helper method to load the CISA KEV catalog to flag known exploited vulnerabilities. This will become important later when we prioritize vulnerabilities based on real-world exploit activity.

In [2]:
import requests

def update_cisa_kev(url=None):
    print("Fetching CISA KEV Catalog...")
    response = requests.get(url)
    if response.status_code != 200:
        print("Failed to fetch CISA KEV data.")
        return
    
    data = response.json()
    vulnerabilities = data.get("vulnerabilities", [])
    
    query = """
    UNWIND $batch AS item
    // Match the existing CVE in your DB
    MATCH (v:CVE {id: item.cveID})
    
    // Add CISA specific metadata
    SET v.kev_addedDate = item.dateAdded,
        v.kev_dueDate = item.dueDate,
        v.kev_reason = item.shortDescription,
        v.isKnownExploited = true
    
    // Create a reference node for the Catalog itself
    MERGE (cat:Catalog {name: 'CISA KEV'})
    SET cat.lastUpdated = $timestamp,
        cat.version = $version
    
    MERGE (v)-[:LISTED_IN]->(cat)
    """
    
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        with driver.session(database=DB) as session:
            session.run(query, 
                        batch=vulnerabilities, 
                        timestamp=data.get('dateReleased'),
                        version=data.get('catalogVersion'))
            
    print(f"Successfully linked {len(vulnerabilities)} exploited vulnerabilities to the CISA KEV.")

### Running the Data Loader

We now proceed to load the data into Neo4j using the defined architecture and data flow. This involves creating the necessary database constraints, parsing the data files, and executing the Cypher queries to populate the graph. In our case we load CVE data between 2000 and 2025 - you might choose to limit this range based on your organization's needs.

In [3]:
create_constraints()

In [4]:
process_local_repo([
    2000, 2001, 2002, 2003, 2004,
    2005, 2006, 2007, 2008, 2009,
    2010, 2011, 2012, 2013, 2014,
    2015, 2016, 2017, 2018, 2019,
    2020, 2021, 2022, 2023, 2024,
    2025
    ])

Scanning directory for files...


Ingesting CVEs:   0%|          | 0/323749 [00:00<?, ?cve/s]

With the main CVE data loaded, we can now enrich our graph with the CISA KEV catalog to flag known exploited vulnerabilities.

In [5]:
update_cisa_kev(KEV_URL)

Fetching CISA KEV Catalog...
Successfully linked 1484 exploited vulnerabilities to the CISA KEV.


### Creating Sample Infrastructure Data

We now create sample infrastructure data representing applications, libraries, build artifacts, and cloud resources. This synthetic data will help us demonstrate the VPEM graph's capabilities in a controlled environment. We have chosen two scenarios: one high-risk path with internet exposure and a low-risk internal application.

Depending on your environment, you would model your actual infrastructure differently, but the principles remain the same.

In [ ]:
def create_vpem_data():
    query = """
        // Setup Libraries
        MERGE (l1:Library {name: 'log4j-core', version: '2.14.1', language: 'Java'})
        MERGE (l2:Library {name: 'spring-web', version: '5.3.8', language: 'Java'})
        MERGE (l3:Library {name: 'openssl', version: '3.0.1', language: 'C'})
        
        // Scenario A: The High-Risk Path (Internet -> App -> S3)
        MERGE (repoA:Repo {name: 'customer-api-prod', criticality: 'High'})
        MERGE (artA:BuildArtifact {id: 'art-prod-001', registry: 'jfrog-prod'})
        MERGE (appA:Application {name: 'CustomerFacingAPI', tier: 'P1'})
        MERGE (insA:ComputeInstance {id: 'i-0001', name: 'api-gateway-01', public_ip: '34.201.1.5'})
        MERGE (endA:Endpoint {url: 'api.acme.com'})
        MERGE (idA:Identity {name: 'service-account-prod-s3', arn: 'arn:aws:iam::123:role/S3Access'})
        MERGE (polA:IAMPolicy {name: 'DataLakeFullAccess'})
        MERGE (cloudA:CloudService {name: 'S3-Bucket', resource_name: 'acme-customer-pii-data'})
        MERGE (teamA:Team {name: 'Platform-Security'})

        MERGE (l1)-[:DEPENDENCY_OF]->(artA)
        MERGE (artA)-[:BUILT_FROM]->(repoA)
        MERGE (artA)-[:RUNNING_AS]->(appA)
        MERGE (appA)-[:HOSTED_ON]->(insA)
        MERGE (endA)-[:RESOLVES_TO]->(insA)
        MERGE (insA)-[:RUNS_AS]->(idA)
        MERGE (appA)-[:AUTHENTICATES_VIA]->(idA)
        MERGE (idA)-[:ASSUMES]->(polA)
        MERGE (polA)-[:HAS_ACCESS_TO]->(cloudA)
        MERGE (teamA)-[:MANAGES]->(idA)

        // Scenario B: The Low-Risk Path (Internal / Isolated)
        MERGE (repoB:Repo {name: 'internal-parser', criticality: 'Low'})
        MERGE (artB:BuildArtifact {id: 'art-dev-999'})
        MERGE (appB:Application {name: 'LegacyParser', tier: 'P3'})
        MERGE (insB:ComputeInstance {id: 'i-0002', name: 'internal-worker-01'})
        
        MERGE (l1)-[:DEPENDENCY_OF]->(artB) 
        MERGE (artB)-[:BUILT_FROM]->(repoB)
        MERGE (artB)-[:RUNNING_AS]->(appB)
        MERGE (appB)-[:HOSTED_ON]->(insB)

        // Scenario C: The "Drop Everything" Path (Recent KEV + Public)
        // This simulates a brand new threat discovered in the last 90 days
        MERGE (repoC:Repo {name: 'edge-auth-provider', criticality: 'Critical'})
        MERGE (artC:BuildArtifact {id: 'art-edge-666'})
        MERGE (appC:Application {name: 'EdgeAuthenticator', tier: 'P1'})
        MERGE (insC:ComputeInstance {id: 'i-0888', name: 'edge-auth-01', public_ip: '52.14.99.1'})
        MERGE (endC:Endpoint {url: 'auth.acme.com'})

        MERGE (l3)-[:DEPENDENCY_OF]->(artC)
        MERGE (artC)-[:BUILT_FROM]->(repoC)
        MERGE (artC)-[:RUNNING_AS]->(appC)
        MERGE (appC)-[:HOSTED_ON]->(insC)
        MERGE (endC)-[:RESOLVES_TO]->(insC)

        // Connect to Existing NVD Data (Mapping via CVE id)
        // l1 gets the famous older Log4j vulns
        WITH l1, l3
        MATCH (v_old:CVE) 
        WHERE v_old.id IN ['CVE-2021-44228', 'CVE-2021-45046']
        MERGE (v_old)-[:IDENTIFIED_IN]->(l1)

        // l3 gets a modern CVE (ensure this ID exists in your loaded NVD/KEV data)
        WITH l3
        MATCH (v_new:CVE)
        WHERE v_new.id IN ['CVE-2023-52163', 'CVE-2025-24990'] // Use real IDs from your current year ingest
        MERGE (v_new)-[:IDENTIFIED_IN]->(l3)
        """
    
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        with driver.session(database=DB) as session:
            session.run(query)

    print("VPEM data creation complete with Recent KEV scenario.")

In [7]:
create_vpem_data()

VPEM data creation complete with Recent KEV scenario.


With all the data loaded, we can now [proceed to analyze and prioritize vulnerabilities using the VPEM graph](vpem.ipynb).